# openapi_spec_validator smoke test

Validates that `openapi_spec_validator` imports cleanly and exercises its primary API surface.

In [ ]:
# Example 1: Validate a minimal valid OpenAPI 3.0 spec using validate().
from openapi_spec_validator import validate

spec = {
    "openapi": "3.0.0",
    "info": {"title": "Test API", "version": "1.0.0"},
    "paths": {
        "/ping": {
            "get": {
                "responses": {
                    "200": {"description": "pong"}
                }
            }
        }
    },
}

# validate() returns None on success; raises on failure.
result = validate(spec)
print(f"validate() returned: {result!r}")
print("Minimal OpenAPI 3.0 spec validated successfully.")

In [ ]:
# Example 2: Detect an invalid OpenAPI spec — should raise a validation error.
from openapi_spec_validator import validate
from openapi_spec_validator.validation.exceptions import OpenAPIValidationError

# Missing required 'info' and 'paths' keys.
bad_spec = {"openapi": "3.0.0"}

try:
    validate(bad_spec)
except OpenAPIValidationError as e:
    print(f"Caught expected {type(e).__name__}: {str(e)[:200]}")
else:
    raise AssertionError("Expected OpenAPIValidationError on malformed spec")

In [ ]:
# Example 3: Use the explicit V30 validator class.
from openapi_spec_validator import OpenAPIV30SpecValidator

spec = {
    "openapi": "3.0.3",
    "info": {"title": "V30 API", "version": "2.0.0"},
    "paths": {},
}

validator = OpenAPIV30SpecValidator(spec)
errors = list(validator.iter_errors())
print(f"Validator class: {type(validator).__name__}")
print(f"Number of errors: {len(errors)}")
assert len(errors) == 0, f"Unexpected errors: {errors}"

In [ ]:
# Example 4: Use the V31 validator and iter_errors() on an invalid spec.
from openapi_spec_validator import OpenAPIV31SpecValidator

# Missing required 'info' field for OpenAPI 3.1.
bad_spec = {
    "openapi": "3.1.0",
    "paths": {},
}

validator = OpenAPIV31SpecValidator(bad_spec)
errors = list(validator.iter_errors())
print(f"Validator class: {type(validator).__name__}")
print(f"Number of errors collected: {len(errors)}")
for err in errors[:3]:
    print(f"  - {type(err).__name__}: {str(err)[:120]}")

assert len(errors) > 0, "Expected at least one validation error"

In [ ]:
# Example 5: Validate a spec with $ref using validate() (resolves references).
from openapi_spec_validator import OpenAPIV30SpecValidator, validate

spec = {
    "openapi": "3.0.0",
    "info": {"title": "Ref API", "version": "1.0.0"},
    "paths": {
        "/items": {
            "get": {
                "responses": {
                    "200": {
                        "description": "ok",
                        "content": {
                            "application/json": {
                                "schema": {"$ref": "#/components/schemas/Item"}
                            }
                        },
                    }
                }
            }
        }
    },
    "components": {
        "schemas": {
            "Item": {"type": "object", "properties": {"id": {"type": "integer"}}}
        }
    },
}

validate(spec)
ref_errors = list(OpenAPIV30SpecValidator(spec).iter_errors())
assert not ref_errors, f"Unexpected errors: {ref_errors}"
print("Spec with $ref validated successfully.")